In [1]:
# 03b_modeling_ridge.ipynb

import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

# 데이터 불러오기 (정확한 변수명!)
with open("X_train.pkl", "rb") as f:
    X_train = pickle.load(f)
with open("y_train.pkl", "rb") as f:
    y_train = pickle.load(f)
with open("X_test.pkl", "rb") as f:
    X_test = pickle.load(f)
with open("y_test.pkl", "rb") as f:
    y_test = pickle.load(f)

# Ridge 모델 + 하이퍼파라미터 튜닝
ridge = Ridge()
params = {'alpha': [0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(ridge, params, cv=5, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

# 최고 모델 저장
best_ridge = grid.best_estimator_

with open("ridge_model.pkl", "wb") as f:
    pickle.dump(best_ridge, f)

# 예측
y_pred = best_ridge.predict(X_test)

# 평가
import numpy as np
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Best alpha: {grid.best_params_['alpha']}")
print(f"RMSE: {rmse:.2f}")
print(f"R^2: {r2:.2f}")

# 결과 저장
pd.DataFrame({
    "actual": y_test,
    "predicted": y_pred
}).to_csv("predictions_ridge.csv", index=False)


Best alpha: 0.01
RMSE: 4.93
R^2: 0.67


In [2]:
# 계수 벡터 (weights)
print(best_ridge.coef_)

# 절편 (bias term)
print(best_ridge.intercept_)

print(f"Best alpha: {grid.best_params_['alpha']}")


[-1.12985398e-01  3.01477290e-02  3.97929679e-02  2.78084775e+00
 -1.70656728e+01  4.43960807e+00 -6.40468889e-03 -1.44590837e+00
  2.62172574e-01 -1.06610344e-02 -9.13880463e-01  1.23566074e-02
 -5.08818863e-01]
30.152156775379012
Best alpha: 0.01


In [3]:
import pandas as pd

coefs = pd.Series(best_ridge.coef_, index=X_test.columns)
coefs = coefs.sort_values()

print(coefs)  # 표 형태로 계수 보기


NOX       -17.065673
DIS        -1.445908
PTRATIO    -0.913880
LSTAT      -0.508819
CRIM       -0.112985
TAX        -0.010661
AGE        -0.006405
B           0.012357
ZN          0.030148
INDUS       0.039793
RAD         0.262173
CHAS        2.780848
RM          4.439608
dtype: float64
